# DOCX 纯文字清洗 · handbook-1p

**样本**：美国司法部 Trustee Handbook 第 1 章（约 3500 字，无表格无图片）

**方法**：python-docx 读正文段落 → 去空段 → Tab/空白规范化

**本 notebook 会处理**：
- 正文段落提取与清洗
- 页眉页脚单独列出（默认不混入正文）

**依赖**：`pip install python-docx`

In [ ]:
import re
from dataclasses import dataclass
from pathlib import Path

from docx import Document


@dataclass
class CleanParagraph:
    """清洗后的一段正文。"""
    index: int          # 在正文中的序号（从 0 开始）
    text: str           # 清洗后的文字
    raw_text: str       # 原始文字（方便对比）


def clean_text(text: str) -> str:
    """清洗单段：Tab 转空格，合并连续空白，去首尾空白。"""
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_headers_footers(doc: Document) -> dict[str, list[str]]:
    """单独提取页眉、页脚文字（不参与正文清洗）。"""
    headers: list[str] = []
    footers: list[str] = []
    for section in doc.sections:
        for para in section.header.paragraphs:
            t = clean_text(para.text)
            if t:
                headers.append(t)
        for para in section.footer.paragraphs:
            t = clean_text(para.text)
            if t:
                footers.append(t)
    return {"headers": headers, "footers": footers}


def clean_docx(docx_path: Path) -> tuple[list[CleanParagraph], dict[str, list[str]]]:
    """读取 docx，返回清洗后的正文段落 + 页眉页脚。"""
    doc = Document(docx_path)
    meta = extract_headers_footers(doc)

    cleaned: list[CleanParagraph] = []
    for para in doc.paragraphs:
        raw = para.text
        text = clean_text(raw)
        if not text:
            continue
        cleaned.append(
            CleanParagraph(index=len(cleaned), text=text, raw_text=raw)
        )
    return cleaned, meta

In [ ]:
# 样本与 notebook 在同一目录，直接用相对路径
# handbook-1p.docx / category-level.docx / example-list-items-multiple.docx
docx_path = Path("category-level.docx")
paragraphs, header_footer = clean_docx(docx_path)

print(f"文件: {docx_path.name}")
print(f"正文段落: {len(paragraphs)} 段")
print(f"总字符: {sum(len(p.text) for p in paragraphs)}")
print(f"页眉: {header_footer['headers']}")
print(f"页脚: {header_footer['footers']}")

In [ ]:
print("=" * 60)
print("清洗后的正文")
print("=" * 60)
for p in paragraphs:
    print(f"\n[{p.index}] {p.text}")

In [ ]:
# 看看清洗前后有没有变化（Tab、多余空白等）
changed = [p for p in paragraphs if p.raw_text.strip() != p.text]
print(f"有变化的段落: {len(changed)} / {len(paragraphs)}")
for p in changed[:5]:
    print(f"\n--- 段 {p.index} ---")
    print(f"原始: {repr(p.raw_text)}")
    print(f"清洗: {p.text}")

---

## 方法二：unstructured

**和 python-docx 的区别**：
- 不用自己读页眉页脚，库会自动标成 `Header` / `Footer`
- 会自动区分短文本（`Text`）和长正文（`NarrativeText`）
- 分页符会标成 `PageBreak`

**依赖**：`pip install "unstructured[docx]"`（项目环境里已安装）

In [20]:
from unstructured.partition.docx import partition_docx


# 正文类型：标题式短句 + 长段落
BODY_TYPES = {"Text", "NarrativeText", "Title", "ListItem"}
# 元信息类型：不进正文
META_TYPES = {"Header", "Footer", "PageBreak"}


@dataclass
class UnstructuredElement:
    """unstructured 解析出的一个元素。"""
    index: int
    category: str   # Header / Text / NarrativeText / Footer ...
    text: str       # 清洗后的文字


def clean_with_unstructured(docx_path: Path) -> tuple[list[UnstructuredElement], list[UnstructuredElement]]:
    """用 unstructured 解析 docx，返回 (正文元素, 元信息元素)。"""
    raw_elements = partition_docx(str(docx_path))

    body: list[UnstructuredElement] = []
    meta: list[UnstructuredElement] = []

    for el in raw_elements:
        text = clean_text(str(el))
        if not text:
            continue
        item = UnstructuredElement(
            index=len(body) + len(meta),
            category=type(el).__name__,
            text=text,
        )
        if type(el).__name__ in META_TYPES:
            meta.append(item)
        elif type(el).__name__ in BODY_TYPES:
            body.append(item)

    return body, meta

In [21]:
# handbook-1p.docx / category-level.docx / example-list-items-multiple.docx
docx_path = Path("example-list-items-multiple.docx")
us_body, us_meta = clean_with_unstructured(docx_path)

print(f"文件: {docx_path.name}")
print(f"正文元素: {len(us_body)} 个")
print(f"总字符: {sum(len(e.text) for e in us_body)}")
print(f"元信息: {len(us_meta)} 个")
for m in us_meta:
    print(f"  [{m.category}] {m.text}")

文件: example-list-items-multiple.docx
正文元素: 14 个
总字符: 275
元信息: 0 个


In [22]:
print("=" * 60)
print("unstructured 正文（带类型标签）")
print("=" * 60)
for i, e in enumerate(us_body):
    print(f"\n[{i}] ({e.category}) {e.text}")

unstructured 正文（带类型标签）

[0] (Text) 第一章 入职准备

[1] (ListItem) 携带身份证、学历证明原件，到人事部办理入职登记。

[2] (ListItem) 领取工牌、电脑和办公用品，并完成邮箱与系统账号开通。

[3] (Text) 第二章 日常工作规范

[4] (ListItem) 工作日按时打卡，迟到超过 30 分钟需向主管说明原因。

[5] (ListItem) 请假须提前一天在系统中提交申请，获批后方可休假。

[6] (ListItem) 病假需提供医院证明；事假按公司制度处理。

[7] (ListItem) 年假天数按工龄计算，具体标准见员工手册。

[8] (Text) 第三章 常见问题

[9] (Text) 以下是新员工最常咨询的三类事项，供快速查阅。

[10] (ListItem) 午餐时间为 12:00 至 13:00，一楼食堂可用餐。

[11] (ListItem) 加班餐补须填写加班申请并经主管审批。

[12] (ListItem) 审批通过后，餐补在次月工资中一并发放。

[13] (ListItem) 未提前审批的加班不计入餐补，请勿事后补填。


In [ ]:
# 和 python-docx 版对比：正文总字符数
py_chars = sum(len(p.text) for p in paragraphs)
us_chars = sum(len(e.text) for e in us_body)
print(f"python-docx  正文: {len(paragraphs)} 段, {py_chars} 字")
print(f"unstructured 正文: {len(us_body)} 个, {us_chars} 字")
print(f"字符差: {abs(py_chars - us_chars)}")

---

## RAG 二次分块：基于 unstructured 分类

**思路**：直接使用 `unstructured` 已经判好的元素类型：

- `Text` → 标题式短句，作为 `section` 标签，不单独成块
- `NarrativeText` → 正文，按 `chunk_size` 切块，挂上当前 section
- `Header` / `Footer` / `PageBreak` → 跳过

这样做的好处：
- 不写任何针对单个文档的硬编码规则（比如「带 A./B.」或「全大写」）
- 逻辑完全依赖库的通用分类，对大多数纯文字 docx 都适用
- 标题识别交给 `unstructured` 的 `is_possible_narrative_text` 等内置函数

| 参数 | 说明 |
|------|------|
| `chunk_size` | 每块最大字数（如 400） |
| `overlap` | 相邻块重复字数（如 80）|

In [23]:
from dataclasses import dataclass, field


@dataclass
class RagChunk:
    """RAG 用的一块数据，带元信息方便检索。"""
    text: str                       # 块内文字
    source_file: str                # 来源文件名
    section: str                    # 所属章节/标题（从 unstructured 的 Text 元素提取）
    chunk_index: int                # 在本文档中的块序号
    char_count: int = field(init=False)

    def __post_init__(self):
        self.char_count = len(self.text)


def chunk_unstructured_elements(
    elements: list[UnstructuredElement],
    source_file: str,
    chunk_size: int = 400,
    overlap: int = 80,
) -> list[RagChunk]:
    """
    基于 unstructured 分类结果做 RAG 分块。

    规则：
    - Text / Title（标题类）→ 更新 current_section，不单独成块
    - NarrativeText / ListItem（正文类）→ 累积到缓冲区，按 chunk_size 切块
    - Header / Footer / PageBreak（元信息类）→ 跳过
    """
    chunks: list[RagChunk] = []
    current_section = "General"
    buffer = ""  # 正文累积缓冲区

    # 标题类型：只更新 section，不进入正文
    HEADING_TYPES = {"Text", "Title"}
    # 正文类型：累积到缓冲区
    BODY_TYPES = {"NarrativeText", "ListItem"}
    # 元信息类型：直接跳过
    META_TYPES = {"Header", "Footer", "PageBreak"}

    def flush_buffer():
        """把缓冲区内容按 chunk_size 切分并加入结果。"""
        nonlocal buffer
        if not buffer:
            return

        if len(buffer) <= chunk_size:
            chunks.append(RagChunk(
                text=buffer.strip(),
                source_file=source_file,
                section=current_section,
                chunk_index=len(chunks),
            ))
        else:
            start = 0
            while start < len(buffer):
                end = min(start + chunk_size, len(buffer))
                chunk_text = buffer[start:end].strip()
                if chunk_text:
                    chunks.append(RagChunk(
                        text=chunk_text,
                        source_file=source_file,
                        section=current_section,
                        chunk_index=len(chunks),
                    ))
                start = end - overlap if end < len(buffer) else end

        buffer = ""

    for el in elements:
        category = el.category

        # 标题类：更新 section，之前的缓冲区先切块
        if category in HEADING_TYPES:
            flush_buffer()
            current_section = el.text
            continue

        # 元信息类：直接跳过
        if category in META_TYPES:
            continue

        # 正文类：累积到缓冲区
        if category in BODY_TYPES:
            buffer += el.text + "\n"
            # 缓冲区满了就切一块
            if len(buffer) >= chunk_size:
                flush_buffer()

    # 最后清空缓冲区
    flush_buffer()

    return chunks

In [24]:
# 用 unstructured 结果做 RAG 分块
rag_chunks = chunk_unstructured_elements(
    us_body,                    # 只有 NarrativeText + Text
    source_file="handbook-1p.docx",
    chunk_size=400,
    overlap=80,
)

print(f"unstructured 元素: {len(us_body)} 个")
print(f"RAG 分块后: {len(rag_chunks)} 块")
print(f"总字符: {sum(c.char_count for c in rag_chunks)}")
print()
print("=" * 60)
print("各块详情（前 10 块）")
print("=" * 60)
for c in rag_chunks[:10]:
    preview = c.text[:60] + "..." if len(c.text) > 60 else c.text
    print(f"\n[{c.chunk_index:2d}] section={c.section[:28]:28s} len={c.char_count:4d}")
    print(f"    {preview}")

unstructured 元素: 14 个
RAG 分块后: 3 块
总字符: 234

各块详情（前 10 块）

[ 0] section=第一章 入职准备                     len=  51
    携带身份证、学历证明原件，到人事部办理入职登记。
领取工牌、电脑和办公用品，并完成邮箱与系统账号开通。

[ 1] section=第二章 日常工作规范                   len=  94
    工作日按时打卡，迟到超过 30 分钟需向主管说明原因。
请假须提前一天在系统中提交申请，获批后方可休假。
病假需提供医院...

[ 2] section=以下是新员工最常咨询的三类事项，供快速查阅。       len=  89
    午餐时间为 12:00 至 13:00，一楼食堂可用餐。
加班餐补须填写加班申请并经主管审批。
审批通过后，餐补在次月工...


In [ ]:
# 统计 section 分布，看 unstructured 把哪些 Text 当成了标题
from collections import Counter

sections = [c.section for c in rag_chunks]
section_counts = Counter(sections)

print("=" * 60)
print("section 分布（unstructured 识别的标题）")
print("=" * 60)
for sec, count in section_counts.most_common():
    print(f"  {sec:35s} → {count:2d} 块")

print()
print("=" * 60)
print("块长度分布")
print("=" * 60)
lengths = [c.char_count for c in rag_chunks]
print(f"最小: {min(lengths):4d} 字")
print(f"最大: {max(lengths):4d} 字")
print(f"平均: {sum(lengths)//len(lengths):4d} 字")
print(f"分布: {sorted(set(lengths))}")